In [1]:
import os
!git clone https://github.com/bencejdanko/bert-tweeteval

# ensure latest
os.chdir('/content/bert-tweeteval')
!cd /content/bert-tweeteval && git pull

Cloning into 'bert-tweeteval'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 136 (delta 81), reused 84 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 175.84 KiB | 359.00 KiB/s, done.
Resolving deltas: 100% (81/81), done.
Already up to date.


In [2]:
# copy over package
PACKAGE = "src"

import sys
sys.path.append(f"/content/bert-tweeteval/{PACKAGE}")

In [3]:
from download import download_and_split_dataset
from analysis import print_samples, print_distribution, calculate_ece
from zero_shot import DistilBERT_zero_shot_pipeline, DistilRoBERTa_zero_shot_pipeline, run_benchmarked_inference, run_zero_shot

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Failed to determine 'entailment' label id from the label2id mapping in the model config. Setting to -1. Define a descriptive label2id mapping in the model config to ensure correct outputs.


In [4]:
id_labels = {0: "anger", 1: "joy", 2: "optimism", 3: "sadness"}
labels_id = {"anger": 0, "joy": 1, "optimism": 2, "sadness": 3}
candidate_labels = list(id_labels.values())
hypothesis_template = "This tweet expresses {}."

In [5]:
train_df, val_df, test_df = download_and_split_dataset()
print(f"Training set: {len(train_df)}")
print(f"Testing set: {len(test_df)}")
print(f"Val set: {len(val_df)}")

README.md: 0.00B [00:00, ?B/s]

emotion/train-00000-of-00001.parquet:   0%|          | 0.00/233k [00:00<?, ?B/s]

emotion/test-00000-of-00001.parquet:   0%|          | 0.00/105k [00:00<?, ?B/s]

emotion/validation-00000-of-00001.parque(…):   0%|          | 0.00/28.6k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3257 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1421 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/374 [00:00<?, ? examples/s]

Training set: 3257
Testing set: 1421
Val set: 374


In [6]:
print_samples(train_df, id_labels)

,Label,Tweet Text
504,joy,second day on the job and i already got a 45 d...
1089,sadness,"@user sadly, at least 3 more years of this. At..."
1452,joy,This white lady just told me that I incite sus...
184,joy,Lady gaga fucking followed i have been waiting...
2112,anger,@user nightmare before Christmas


In [7]:
print_distribution(train_df, test_df, id_labels)


Data Split
Training set: 3257
Testing set:  1421

Class Distribution (Counts)
          Train Count  Test Count  Train %
label                                     
anger            1400         558    42.98
sadness           855         382    26.25
joy               708         358    21.74
optimism          294         123     9.03


In [8]:
test_df["distilbert_pred"] = run_zero_shot(test_df, DistilBERT_zero_shot_pipeline, candidate_labels , hypothesis_template)

In [9]:
test_df["distilroberta_pred"] = run_zero_shot(test_df, DistilRoBERTa_zero_shot_pipeline, candidate_labels, hypothesis_template)

In [10]:
print(test_df[["text", "label", "distilbert_pred", "distilroberta_pred"]].head())

                                                text  label distilbert_pred  \
0  #Deppression is real. Partners w/ #depressed p...      3           anger   
1  @user Interesting choice of words... Are you c...      0           anger   
2  My visit to hospital for care triggered #traum...      3           anger   
3  @user Welcome to #MPSVT! We are delighted to h...      1           anger   
4                       What makes you feel #joyful?      1           anger   

  distilroberta_pred  
0              anger  
1              anger  
2              anger  
3              anger  
4              anger  


In [11]:
results = {}

In [12]:
results["DistilBERT (WordPiece)"] = run_benchmarked_inference(test_df, DistilBERT_zero_shot_pipeline, candidate_labels, labels_id, hypothesis_template)

In [13]:
results["DistilRoBERTa (BPE)"] = run_benchmarked_inference(test_df, DistilRoBERTa_zero_shot_pipeline, candidate_labels, labels_id, hypothesis_template)

In [15]:
import pandas as pd
key_metrics = ["Accuracy", "Macro F1", "ECE", "Time/100"]
pd.DataFrame({
    model_name: {metric: results[model_name][metric] for metric in key_metrics}
    for model_name in results.keys()
}).transpose()

,Accuracy,Macro F1,ECE,Time/100
DistilBERT (WordPiece),0.396904,0.167962,0.145010,0.324951
DistilRoBERTa (BPE),0.389163,0.140284,0.138905,0.310077


In [16]:
pd.DataFrame({
    model_name: {metric: results[model_name][metric] for metric in key_metrics}
    for model_name in results.keys()
}).transpose().to_markdown()

'|                        |   Accuracy |   Macro F1 |      ECE |   Time/100 |\n|:-----------------------|-----------:|-----------:|---------:|-----------:|\n| DistilBERT (WordPiece) |   0.396904 |   0.167962 | 0.14501  |   0.324951 |\n| DistilRoBERTa (BPE)    |   0.389163 |   0.140284 | 0.138905 |   0.310077 |'

In [17]:
import pandas as pd
report_df = pd.DataFrame(results["DistilBERT (WordPiece)"]["Classification Report Dict"]).transpose()
report_df

,precision,recall,f1-score,support
anger,0.395517,0.980287,0.563627,558.000000
joy,0.454545,0.041899,0.076726,358.000000
optimism,0.500000,0.016260,0.031496,123.000000
sadness,0.000000,0.000000,0.000000,382.000000
accuracy,0.396904,0.396904,0.396904,0.396904
macro avg,0.337516,0.259612,0.167962,1421.000000
weighted avg,0.313107,0.396904,0.243382,1421.000000


In [18]:
report_df.to_markdown()

'|              |   precision |    recall |   f1-score |     support |\n|:-------------|------------:|----------:|-----------:|------------:|\n| anger        |    0.395517 | 0.980287  |  0.563627  |  558        |\n| joy          |    0.454545 | 0.0418994 |  0.0767263 |  358        |\n| optimism     |    0.5      | 0.0162602 |  0.0314961 |  123        |\n| sadness      |    0        | 0         |  0         |  382        |\n| accuracy     |    0.396904 | 0.396904  |  0.396904  |    0.396904 |\n| macro avg    |    0.337516 | 0.259612  |  0.167962  | 1421        |\n| weighted avg |    0.313107 | 0.396904  |  0.243382  | 1421        |'

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_true = test_df['label'].map(id_labels)
y_pred_bert = test_df['distilbert_pred']
cm_bert = confusion_matrix(y_true, y_pred_bert, labels=candidate_labels)
disp_bert = ConfusionMatrixDisplay(confusion_matrix=cm_bert, display_labels=candidate_labels)
disp_bert.plot(cmap=plt.cm.Blues)
plt.title("DistilBERT (WordPiece) Confusion Matrix")
plt.show()

In [19]:
import pandas as pd
report_df = pd.DataFrame(results["DistilRoBERTa (BPE)"]["Classification Report Dict"]).transpose()
report_df

,precision,recall,f1-score,support
anger,0.391366,0.991039,0.561136,558.000000
joy,0.000000,0.000000,0.000000,358.000000
optimism,0.000000,0.000000,0.000000,123.000000
sadness,0.000000,0.000000,0.000000,382.000000
accuracy,0.389163,0.389163,0.389163,0.389163
macro avg,0.097841,0.247760,0.140284,1421.000000
weighted avg,0.153682,0.389163,0.220348,1421.000000


In [20]:
report_df.to_markdown()

'|              |   precision |   recall |   f1-score |     support |\n|:-------------|------------:|---------:|-----------:|------------:|\n| anger        |   0.391366  | 0.991039 |   0.561136 |  558        |\n| joy          |   0         | 0        |   0        |  358        |\n| optimism     |   0         | 0        |   0        |  123        |\n| sadness      |   0         | 0        |   0        |  382        |\n| accuracy     |   0.389163  | 0.389163 |   0.389163 |    0.389163 |\n| macro avg    |   0.0978415 | 0.24776  |   0.140284 | 1421        |\n| weighted avg |   0.153682  | 0.389163 |   0.220348 | 1421        |'

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_true = test_df['label'].map(id_labels)
y_pred_rob = test_df['distilroberta_pred']
cm_rob = confusion_matrix(y_true, y_pred_rob, labels=candidate_labels)
disp_rob = ConfusionMatrixDisplay(confusion_matrix=cm_rob, display_labels=candidate_labels)
disp_rob.plot(cmap=plt.cm.Blues)
plt.title("DistilRoBERTa (BPE) Confusion Matrix")
plt.show()

In [23]:
import pandas as pd
from analysis import show_tokenization

both_mis_df = test_df[
    (test_df['label'] != test_df['distilbert_pred'].map(labels_id)) &
    (test_df['label'] != test_df['distilroberta_pred'].map(labels_id))
].copy()

zero_shot_error_samples = both_mis_df.sample(8, random_state=15179996)

zero_shot_error_samples.to_csv("zero_shot_errors.csv", index=True)

print(f"Found {len(both_mis_df)} samples misclassified by both models in zero-shot.")
print("Displaying 8 samples with tokenization splits:\n")

for i, row in zero_shot_error_samples.iterrows():
    print(f"True Label: {id_labels[row['label']]}")
    print(f"DistilBERT pred: {row['distilbert_pred']}")
    print(f"DistilRoBERTa pred: {row['distilroberta_pred']}")
    print(f"Text: {row['text']}")

    tokens_data = show_tokenization(row['text'])
    for d in tokens_data:
        print(f"  {d['Tokenizer']} ({d['Model']}): {d['Tokens']}")
    print()

Found 847 samples misclassified by both models in zero-shot.
Displaying 8 samples with tokenization splits:

True Label: sadness
DistilBERT pred: anger
DistilRoBERTa pred: anger
Text: @user Improve on the makeup dear to avoid reduction in viewership...
  WordPiece (DistilBERT): ['@', 'user', 'improve', 'on', 'the', 'makeup', 'dear', 'to', 'avoid', 'reduction', 'in', 'viewers', '##hip', '.', '.', '.']
  BPE (DistilRoBERTa): ['@', 'user', 'ĠImprove', 'Ġon', 'Ġthe', 'Ġmakeup', 'Ġdear', 'Ġto', 'Ġavoid', 'Ġreduction', 'Ġin', 'Ġviewership', '...']

True Label: sadness
DistilBERT pred: anger
DistilRoBERTa pred: anger
Text: nothing happened to make me sad but i almost burst into tears like 3 times today¿
  WordPiece (DistilBERT): ['nothing', 'happened', 'to', 'make', 'me', 'sad', 'but', 'i', 'almost', 'burst', 'into', 'tears', 'like', '3', 'times', 'today', '¿']
  BPE (DistilRoBERTa): ['nothing', 'Ġhappened', 'Ġto', 'Ġmake', 'Ġme', 'Ġsad', 'Ġbut', 'Ġi', 'Ġalmost', 'Ġburst', 'Ġinto', 'Ġtears', 

In [ ]:
import pandas as pd
from analysis import show_tokenization

both_mis_df = test_df[
    (test_df['label'] != test_df['distilbert_pred'].map(labels_id)) &
    (test_df['label'] != test_df['distilroberta_pred'].map(labels_id))
].copy()

zero_shot_error_samples = both_mis_df.sample(8, random_state=15179996)

zero_shot_error_samples.to_csv("zero_shot_errors.csv", index=True)

print(f"Found {len(both_mis_df)} samples misclassified by both models in zero-shot: \n")

for i, row in zero_shot_error_samples.iterrows():
    print(f"True Label: {id_labels[row['label']]}")
    print(f"DistilBERT pred: {row['distilbert_pred']}")
    print(f"DistilRoBERTa pred: {row['distilroberta_pred']}")
    print(f"Text: {row['text']}")

    tokens_data = show_tokenization(row['text'])
    for d in tokens_data:
        print(f"  {d['Tokenizer']} ({d['Model']}): {d['Tokens']}")
    print()

In [ ]:
import pandas as pd
test_df['label_str'] = test_df['label'].map(id_labels)

both_wrong = test_df[(test_df['distilbert_pred'] != test_df['label_str']) & (test_df['distilroberta_pred'] != test_df['label_str'])].head(8)

bert_right_rob_wrong = test_df[(test_df['distilbert_pred'] == test_df['label_str']) & (test_df['distilroberta_pred'] != test_df['label_str'])].head(8)

rob_right_bert_wrong = test_df[(test_df['distilroberta_pred'] == test_df['label_str']) & (test_df['distilbert_pred'] != test_df['label_str'])].head(8)

print(f"Both wrong: {len(both_wrong)}")
print(f"DistilBERT right, DistilRoBERTa wrong: {len(bert_right_rob_wrong)}")
print(f"DistilRoBERTa right, DistilBERT wrong: {len(rob_right_bert_wrong)}")

both_wrong_indices = both_wrong.index.tolist()
bert_right_rob_wrong_indices = bert_right_rob_wrong.index.tolist()
rob_right_bert_wrong_indices = rob_right_bert_wrong.index.tolist()


In [ ]:
from transformers import AutoTokenizer

tokenizer_bert = DistilBERT_zero_shot_pipeline.tokenizer
tokenizer_rob = DistilRoBERTa_zero_shot_pipeline.tokenizer

def analyze_tokenization(df, title):
    for idx, row in df.iterrows():
        text = row['text']
        true_label = row['label_str']
        pred_bert = row['distilbert_pred']
        pred_rob = row['distilroberta_pred']

        tokens_bert = tokenizer_bert.tokenize(text)
        tokens_rob = tokenizer_rob.tokenize(text)

        print(f"Text: {text}")
        print(f"True: {true_label} | DistilBERT Pred: {pred_bert} | DistilRoBERTa Pred: {pred_rob}")
        print(f"WordPiece (DistilBERT): {tokens_bert}")
        print(f"BPE (DistilRoBERTa): {tokens_rob}")

analyze_tokenization(both_wrong, "Misclassified by BOTH")
analyze_tokenization(bert_right_rob_wrong, "DistilBERT Right, DistilRoBERTa Wrong")
analyze_tokenization(rob_right_bert_wrong, "DistilRoBERTa Right, DistilBERT Wrong")


In [ ]:
import json
with open("error_analysis_indices.json", "w") as f:
    json.dump({
        "both_wrong": both_wrong_indices,
        "bert_right_rob_wrong": bert_right_rob_wrong_indices,
        "rob_right_bert_wrong": rob_right_bert_wrong_indices
    }, f)
print("Saved indices to error_analysis_indices.json")
